# STARVIS Database Explorer

This notebook allows you to explore the local PostgreSQL database for STARVIS.

## Requirements & Setup

To run this notebook, you need to set up a Python virtual environment and install the required dependencies:

```bash
# Initialize and activate the virtual environment in notebooks/
cd notebooks
python -m venv .venv
# On Windows (Powershell):
.venv\Scripts\Activate.ps1

# Install required packages
pip install pandas sqlalchemy psycopg2-binary matplotlib ipykernel
```


In [ ]:
# Optional inline installation of dependencies if they are not already installed
# %pip install pandas sqlalchemy psycopg2-binary matplotlib ipykernel

## Database Connection Setup

We'll connect to the local PostgreSQL database instance using `sqlalchemy`. By default, the database is configured in `.env.dev`.


In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine

# Read DB connection details (using local dev defaults from .env.dev)
db_user = "starvis_user"
db_pass = "starvis_pass"
db_host = "localhost"
db_port = "5432"
db_name = "starvis"

# Connection string
conn_str = f"postgresql://{db_user}:{db_pass}@{db_host}:{db_port}/{db_name}"
engine = create_engine(conn_str)
print("SqlAlchemy connection engine initialized.")

## Monorepo Database Statistics

Let's query the counts of game components, items, commodities, missions, and ships.


In [ ]:
# Query counts from key tables in the game and rsi schemas
queries = {
    "Components (extracted game data)": "SELECT COUNT(*) FROM game.components",
    "Items (extracted FPS gear, etc.)": "SELECT COUNT(*) FROM game.items",
    "Commodities (trade goods)": "SELECT COUNT(*) FROM game.commodities",
    "Missions": "SELECT COUNT(*) FROM game.missions",
    "Ships (matrix)": "SELECT COUNT(*) FROM rsi.ship_matrix"
}

stats = []
for name, sql in queries.items():
    try:
        count = pd.read_sql(sql, engine).iloc[0, 0]
        stats.append({"Entity": name, "Count": count})
    except Exception as e:
        stats.append({"Entity": name, "Count": f"Error: {e}"})

df_stats = pd.DataFrame(stats)
print(df_stats.to_string(index=False))

## Game Component Categories Distribution

Let's query and plot the number of components per category to visualize our extracted database content.


In [ ]:
import matplotlib.pyplot as plt

# Query components grouped by category
sql_categories = """
SELECT category, COUNT(*) as count 
FROM game.components 
GROUP BY category 
ORDER BY count DESC
"""
df_categories = pd.read_sql(sql_categories, engine)

if not df_categories.empty:
    plt.figure(figsize=(12, 6))
    plt.bar(df_categories['category'], df_categories['count'], color='#1f77b4')
    plt.xticks(rotation=45, ha='right')
    plt.xlabel('Category')
    plt.ylabel('Count')
    plt.title('Game Components by Category')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("No components found in the database. Run data extraction first!")